In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from collections import defaultdict

In [2]:
# Inputs from previous CP stage + your sheet workbook
CP_OUTPUT_PATH = Path("../data/Processed Data/Schedule_Output_CP.xlsx")

# Keep Final CP output unchanged by writing to a new file
OUTPUT_PATH = Path("../data/Processed Data/Schedule_Output_S.xlsx")

In [3]:
# Load CP output (input to this stage)
cp_schedule = pd.read_excel(CP_OUTPUT_PATH, sheet_name="Schedule")

# Load your section-related sheets
sdata_path = Path("../data/Raw Data/SData.xlsx")
Rooms = pd.read_excel(sdata_path, sheet_name="Rooms")
Sections = pd.read_excel(sdata_path, sheet_name="Section")
Assistant = pd.read_excel(sdata_path, sheet_name="Assistant")
Divisions = pd.read_excel(sdata_path, sheet_name="Division")

# Load Data workbook for section instructors (course -> instructor)
data_path = Path("../data/Raw Data/Data.xlsx")
CoursesData = pd.read_excel(data_path, sheet_name="Courses")
DoctorsData = pd.read_excel(data_path, sheet_name="Doctors")

print(f"CP rows: {len(cp_schedule)}")
print(f"Sections rows: {len(Sections)}")
print(f"Rooms rows: {len(Rooms)}")

CP rows: 105
Sections rows: 53
Rooms rows: 8


In [4]:
def find_column(df, candidates, required=True):
    lowered = {str(c).strip().lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lowered:
            return lowered[cand.lower()]
    if required:
        raise ValueError(f"Missing required column. Tried: {candidates}. Available: {list(df.columns)}")
    return None

# Infer common columns safely from your sheets
room_name_col = find_column(Rooms, ["Room", "Room_Name"])
room_type_col = find_column(Rooms, ["Type", "Room_Type"], required=False)

sec_course_col = find_column(Sections, ["Course_Name", "Course", "Subject"], required=False)
sec_div_col = find_column(Sections, ["Division", "Num_ID", "Division_ID", "Group_ID", "Group", "Major"], required=False)
sec_name_col = find_column(Sections, ["Section", "Section_Name", "Sec", "Name"], required=False)
sec_inst_col = find_column(Sections, ["Instructor_Name", "Instructor", "Doctor", "Assistant", "Assistant_Name", "TA"], required=False)

assistant_id_col = find_column(Assistant, ["Assistant_ID", "ID", "Instructor_ID", "TA_ID"], required=False)
assistant_name_col = find_column(Assistant, ["Assistant_Name", "Name", "Instructor_Name", "TA_Name"], required=False)
if assistant_name_col is None and len(Assistant.columns) > 0:
    assistant_name_col = Assistant.columns[-1]
if assistant_id_col is None and len(Assistant.columns) > 0:
    assistant_id_col = Assistant.columns[0]

# Assistant ID -> Name lookup (to replace IDs in output)
assistant_id_to_name = {}
if assistant_id_col is not None and assistant_name_col is not None:
    for _, arow in Assistant.iterrows():
        aid = str(arow[assistant_id_col]).strip()
        aname = str(arow[assistant_name_col]).strip()
        if aid and aname:
            assistant_id_to_name[aid] = aname

def normalize_instructor_name(value):
    raw = "" if value is None else str(value).strip()
    return assistant_id_to_name.get(raw, raw)

div_id_col = find_column(Divisions, ["Num_ID", "Division", "Division_ID", "Group_ID"], required=False)
div_students_col = find_column(Divisions, ["StudentNum", "Students", "Student_Count"], required=False)

# Build reusable CP slot table
cp_slots = cp_schedule.copy()
for col in ["Course_Name", "Day", "Start_Time", "End_Time"]:
    if col not in cp_slots.columns:
        raise ValueError(f"CP output missing column: {col}")

for col in ["Assistant_Name", "Major", "Instructor_Name", "Students", "Room"]:
    if col not in cp_slots.columns:
        cp_slots[col] = ""

# Final output columns: Assistant_Name included (Department removed)
final_cols = [
    "Day",
    "Course_Name",
    "Instructor_Name",
    "Assistant_Name",
    "Students",
    "Room",
    "Start_Time",
    "End_Time",
    "Major",
]

# Prefer labs for section scheduling if available
if room_type_col is not None:
    lab_rooms = Rooms[Rooms[room_type_col].astype(str).str.lower().str.contains("lab", na=False)][room_name_col].astype(str).tolist()
else:
    lab_rooms = []
if not lab_rooms:
    lab_rooms = Rooms[room_name_col].astype(str).tolist()

assistant_pool = []
if assistant_name_col is not None:
    assistant_pool = (
        Assistant[assistant_name_col]
        .dropna()
        .astype(str)
        .map(lambda x: normalize_instructor_name(x))
        .tolist()
    )
if not assistant_pool:
    assistant_pool = ["TBA"]

# Lookup for division students
division_students = {}
if div_id_col and div_students_col:
    for _, drow in Divisions.iterrows():
        did = str(drow[div_id_col]).strip()
        try:
            division_students[did] = int(float(drow[div_students_col])) // 2
        except Exception:
            division_students[did] = ""

# Build course -> instructor mapping from Data.xlsx
course_instructor_from_data = {}
if {"Course_Name", "Instructor_ID"}.issubset(CoursesData.columns) and {"Instructor_ID", "Instructor_Name"}.issubset(DoctorsData.columns):
    doctors_lookup = {
        str(r["Instructor_ID"]).strip(): str(r["Instructor_Name"]).strip()
        for _, r in DoctorsData.iterrows()
    }
    for _, r in CoursesData.iterrows():
        cname = str(r["Course_Name"]).strip()
        iid = str(r["Instructor_ID"]).strip()
        iname = doctors_lookup.get(iid, "")
        if cname and iname:
            course_instructor_from_data[cname] = iname

# Lookup for default instructor/major from CP schedule by course
course_defaults = {}
for _, crow in cp_slots.iterrows():
    cname = str(crow.get("Course_Name", "")).strip()
    if cname and cname not in course_defaults:
        course_defaults[cname] = {
            "Instructor_Name": str(crow.get("Instructor_Name", "")).strip(),
            "Major": str(crow.get("Major", "")).strip(),
        }

# Occupancy trackers to avoid clashes
room_busy = set()
instructor_busy = set()
section_busy = set()

section_rows = []
auto_instructor_idx = 0

for i, sec in Sections.iterrows():
    course_name = str(sec[sec_course_col]).strip() if sec_course_col else ""
    division_name = str(sec[sec_div_col]).strip() if sec_div_col else ""
    section_name = str(sec[sec_name_col]).strip() if sec_name_col else f"Section_{i+1}"

    # Assistant shown in dedicated Assistant_Name column
    section_assistant_name = normalize_instructor_name(assistant_pool[i % len(assistant_pool)])

    # For section rows, Instructor_Name should be an assistant.
    # Lecture rows remain unchanged from CP output (typically doctors).
    instructor_name = ""
    if sec_inst_col and pd.notna(sec[sec_inst_col]) and str(sec[sec_inst_col]).strip():
        instructor_name = normalize_instructor_name(sec[sec_inst_col])
    else:
        instructor_name = normalize_instructor_name(assistant_pool[auto_instructor_idx % len(assistant_pool)])
        auto_instructor_idx += 1

    # Prefer matching course slots; fallback to all CP slots
    candidates = cp_slots.copy()
    if course_name:
        by_course = candidates[candidates["Course_Name"].astype(str).str.strip() == course_name]
        if not by_course.empty:
            candidates = by_course
    if division_name:
        by_major = candidates[candidates["Major"].astype(str).str.strip() == division_name]
        if not by_major.empty:
            candidates = by_major

    assigned = False
    for _, slot in candidates.iterrows():
        day = str(slot["Day"]).strip()
        start_time = str(slot["Start_Time"]).strip()
        end_time = str(slot["End_Time"]).strip()

        for room in lab_rooms:
            room_key = (day, start_time, end_time, room)
            inst_key = (day, start_time, end_time, instructor_name)
            sec_key = (day, start_time, end_time, section_name)

            if room_key in room_busy or inst_key in instructor_busy or sec_key in section_busy:
                continue

            room_busy.add(room_key)
            instructor_busy.add(inst_key)
            section_busy.add(sec_key)

            defaults = course_defaults.get(course_name, {})
            students = division_students.get(division_name, "")

            section_rows.append(
                {
                    "Day": day,
                    "Course_Name": course_name if course_name else str(slot["Course_Name"]),
                    "Instructor_Name": instructor_name or defaults.get("Instructor_Name", ""),
                    "Assistant_Name": section_assistant_name,
                    "Students": students,
                    "Room": room,
                    "Start_Time": start_time,
                    "End_Time": end_time,
                    "Major": division_name if division_name else defaults.get("Major", ""),
                }
            )
            assigned = True
            break

        if assigned:
            break

    if not assigned:
        defaults = course_defaults.get(course_name, {})
        students = division_students.get(division_name, "")

        section_rows.append(
            {
                "Day": "UNASSIGNED",
                "Course_Name": course_name,
                "Instructor_Name": instructor_name or defaults.get("Instructor_Name", ""),
                "Assistant_Name": section_assistant_name,
                "Students": students,
                "Room": "",
                "Start_Time": "",
                "End_Time": "",
                "Major": division_name if division_name else defaults.get("Major", ""),
            }
        )

section_schedule = pd.DataFrame(section_rows)

# Keep CP output values exactly as generated in Final CP
cp_formatted = cp_schedule.copy()
if "Assistant_Name" not in cp_formatted.columns:
    cp_formatted["Assistant_Name"] = ""
for col in final_cols:
    if col not in cp_formatted.columns:
        cp_formatted[col] = ""
cp_formatted = cp_formatted[final_cols]

# Section rows populate Assistant_Name directly
combined_schedule = pd.concat([cp_formatted, section_schedule[final_cols]], ignore_index=True)

def write_output_excel(target_path):
    with pd.ExcelWriter(target_path, engine="openpyxl") as writer:
        combined_schedule.to_excel(writer, sheet_name="Schedule", index=False)
        cp_formatted.to_excel(writer, sheet_name="CP_Only", index=False)
        section_schedule.to_excel(writer, sheet_name="Sections_Only", index=False)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
written_path = OUTPUT_PATH

try:
    write_output_excel(OUTPUT_PATH)
except PermissionError:
    # Common on Windows when the target workbook is open in Excel.
    fallback_path = OUTPUT_PATH.with_name(
        f"{OUTPUT_PATH.stem}_{pd.Timestamp.now():%Y%m%d_%H%M%S}{OUTPUT_PATH.suffix}"
    )
    write_output_excel(fallback_path)
    written_path = fallback_path
    print(f"Target file is locked, wrote fallback file instead: {fallback_path.resolve()}")

print(f"Created: {written_path.resolve()}")
print(f"CP rows: {len(cp_formatted)} | Section rows: {len(section_schedule)} | Total rows: {len(combined_schedule)}")
print(combined_schedule.head(10))

Created: C:\Work\ML\Graduation\data\Processed Data\Schedule_Output_S.xlsx
CP rows: 105 | Section rows: 53 | Total rows: 158
         Day                Course_Name  Instructor_Name Assistant_Name  \
0    Tuesday            Mathematics (0)        Dr. Amany                  
1   Saturday            Mathematics (0)        Dr. Amany                  
2    Tuesday            Mathematics (1)        Dr. Amany                  
3   Saturday            Mathematics (1)        Dr. Amany                  
4   Saturday               Electronics       Dr. Ibrahim                  
5    Tuesday               Electronics       Dr. Ibrahim                  
6     Sunday      Discrete Mathematics        Dr. Moataz                  
7  Wednesday      Discrete Mathematics        Dr. Moataz                  
8  Wednesday  Introduction to Computers  Dr. Dina Tarek                   
9     Sunday  Introduction to Computers  Dr. Dina Tarek                   

  Students       Room Start_Time  End_Time Major  